In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from shapely.geometry import Point
from shapely.geometry import box
from shapely.geometry import Polygon

In [ ]:
# load data
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp")
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")
reg = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")
synfire_7d = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/SynFire_7d_unique_id_list_five_levels.csv")

# Plot Synchronous Event Example

In [ ]:
start = "2003-03-21"
end = "2003-03-27"
level = "High"

#"No": start = "2001-01-01", end = "2001-01-07"
#"Regional": start = "2005-06-05", end = "2005-06-11"
#"Low": start = "2002-08-22", end = "2002-08-28"
#"Medium": start = "2003-08-02", end = "2003-08-08"
#"High": start = "2003-03-21", end = "2003-03-27"

In [ ]:
event = plt.figure(figsize = (8, 10))
ax = plt.gca()
#Continental Synchronicity
study_area.plot(ax = ax, facecolor = "#dddddd", edgecolor = "#dddddd")

for _,row in reg.iterrows():
    prbox = row.geometry
    gpd.GeoSeries([prbox]).boundary.plot(ax = ax, color = "black")

#Continental Synchronicity
fire_event_set = fire_obs[(fire_obs["start_date"] >= start) & (fire_obs["start_date"] <= end)]

if len(fire_event_set) != 0:
    #constrcut point geometry
    Y = fire_event_set["LAT"]
    X = fire_event_set["LON"]
        
    points = [(lon, lat) for lon, lat in zip(X, Y)]
    fire_center = gpd.GeoDataFrame(
            {'geometry': [Point(lon, lat) for lon, lat in points]},
            crs="EPSG:4326"
    )
    fire_center.plot(ax = ax, marker='o', color='red', markersize=15)

    #color the regions involved
    reg_set = set(fire_event_set["region"])
    
    for _, row in reg.iterrows():
        prbox = row.geometry
        abbrev = row.abbrev
        if abbrev in reg_set:
            gpd.GeoSeries([prbox]).boundary.plot(ax = ax, color = "red")

ratio = 1.4
x_left, x_right = ax.get_xlim()
y_low, y_high = ax.get_ylim()
ax.set_aspect(abs((x_right-x_left)/(y_low-y_high))*ratio)

#add labels
ax.text(-10, 64, "BI", fontsize = 15)
ax.text(-9, 43, "IP", fontsize = 15)
ax.text(-4, 49, "FR", fontsize = 15)
ax.text(2.5, 54, "ME", fontsize = 15)
ax.text(5, 70.5, "SC", fontsize = 15)
ax.text(6, 47, "AL", fontsize = 15)
ax.text(6, 43, "WMD", fontsize = 15)
ax.text(16.5, 43, "EMD", fontsize = 15)
ax.text(20, 59, "NEA", fontsize = 15)
ax.text(16.5, 49, "SEA", fontsize = 15)

#add title and axis labels
ax.text(0.03, 0.95, level , fontsize = 22, transform = ax.transAxes)
ax.set_xticks([])
ax.set_xticklabels([])
ax.set_yticks([])
ax.set_yticklabels([])

#legend
square =  mlines.Line2D([], [], color='black', marker='s', markersize=15, linestyle='None', fillstyle = 'none', label='Climate Zone', markeredgewidth = 2)
dot = mlines.Line2D([], [], color = 'red', marker = 'o', markersize=5, linestyle = 'None', label = "Fire Event")
ax.legend(handles = [square, dot], 
               loc='upper left', fontsize=13, title='', frameon = False, ncol = 1, bbox_to_anchor=(-0.035, 0.92), handletextpad = 0)

plt.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/Event_example_{level}.png", dpi = 600, bbox_inches = "tight")
plt.show()

## All in one

In [ ]:
fig, axes = plt.subplots(2, 3, figsize = (24, 20))
axes = axes.flatten()

#general settings for axes:
for ax in axes:

    # turn off x and y
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])

    # add region labels
    ax.text(-10, 64, "BI", fontsize = 15)
    ax.text(-9, 43, "IP", fontsize = 15)
    ax.text(-4, 49, "FR", fontsize = 15)
    ax.text(2.5, 54, "CE", fontsize = 15)  #ME->CE
    ax.text(5, 70.5, "SC", fontsize = 15)
    ax.text(6, 47, "AL", fontsize = 15)
    ax.text(6, 43, "WMD", fontsize = 15)
    ax.text(16.5, 43, "EMD", fontsize = 15)
    ax.text(20, 59, "NEE", fontsize = 15)  #NEA->NEE
    ax.text(16.5, 49, "SEE", fontsize = 15)  #SEA->SEE

#----------------------------
#plot fire event distribution
ax = axes[0]

# get fire centers
fire_obs = fire_obs.to_crs(epsg = 3035)
fire_centers = list(zip(fire_obs.geometry.centroid.x, fire_obs.geometry.centroid.y))
gpd_fire_centers = gpd.GeoDataFrame(geometry = [Point(fire_center) for fire_center in fire_centers], crs = "EPSG:3035")
gpd_fire_centers = gpd_fire_centers.to_crs(epsg = 4326)
fire_obs = fire_obs.to_crs(epsg = 4326)

# plot study area, fire events and 10 regions
study_area.plot(ax = ax, facecolor = "#dddddd", edgecolor = "#dddddd")
gpd_fire_centers.plot(ax = ax, markersize = 0.5, marker='o',  color = "red")
reg.boundary.plot(ax = ax, edgecolor = "black")

#add title
ax.text(0.01, 0.96, "(a) Event Distribution" , fontsize = 18, transform = ax.transAxes)

#legend
square =  mlines.Line2D([], [], color='black', marker='s', markersize=15, linestyle='None', fillstyle = 'none', label='Climate Zone', markeredgewidth = 2)
dot = mlines.Line2D([], [], color = 'red', marker = 'o', markersize=1, linestyle = 'None', label = "Fire Event")
ax.legend(handles = [square, dot], 
          loc='upper left', fontsize=15, title='', frameon = False, ncol = 1, bbox_to_anchor=(-0.035, 0.92), handletextpad = 0)

#----------------------------
#plot synchronous fire event example
for date, title, ax in zip(["2001-01-01", "2005-06-05", "2002-08-22", "2003-08-02", "2003-03-21"], ["(b) No", "(c) Regional", "(d) Low", "(e) Medium", "(f) High"], axes[1:]):

    study_area.plot(ax = ax, facecolor = "#dddddd", edgecolor = "#dddddd")
    
    #Continental Synchronicity
    fire_event_set = fire_obs[(fire_obs["start_date"] >= date) & (fire_obs["start_date"] <= (pd.Timestamp(date) + pd.Timedelta(days = 6)).strftime("%Y-%m-%d"))]
    
    if len(fire_event_set) != 0:
        
        fire_event_set = fire_event_set.to_crs(epsg = 3035)
        fire_centers = list(zip(fire_event_set.geometry.centroid.x, fire_event_set.geometry.centroid.y))
        gpd_fire_centers = gpd.GeoDataFrame(geometry = [Point(fire_center) for fire_center in fire_centers], crs = "EPSG:3035")
        gpd_fire_centers = gpd_fire_centers.to_crs(epsg = 4326)
        gpd_fire_centers.plot(ax = ax, marker='o', color='red', markersize=15)
    
        #color the regions involved
        reg_set = set(fire_event_set["region"])
        
        for _, row in reg.iterrows():
            prbox = row.geometry
            abbrev = row.abbrev
            if abbrev in reg_set:
                gpd.GeoSeries([prbox]).boundary.plot(ax = ax, color = "red")
                
    reg.boundary.plot(ax = ax, edgecolor = "black")
    
    
    #add title and axis labels
    ax.text(0.01, 0.96, title , fontsize = 18, transform = ax.transAxes)
    
    #legend
    square =  mlines.Line2D([], [], color='black', marker='s', markersize=15, linestyle='None', fillstyle = 'none', label='Climate Zone', markeredgewidth = 2)
    dot = mlines.Line2D([], [], color = 'red', marker = 'o', markersize=5, linestyle = 'None', label = "Fire Event")
    ax.legend(handles = [square, dot], 
                   loc='upper left', fontsize=15, title='', frameon = False, ncol = 1, bbox_to_anchor=(-0.035, 0.92), handletextpad = 0)

plt.subplots_adjust(hspace = 0.01, wspace = -0.25)

#save
plt.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/S6_Event_Distribution_and_Five_levels_synchronous_fire_event_examples.png", dpi = 600, bbox_inches = "tight")